![Henry Logo](https://www.soyhenry.com/_next/static/media/HenryLogo.bb57fd6f.svg)


# Introducción a Transformers Architecture


* Profesor : [Carlos Daniel Jiménez Martínez](danieljimenez88m@gmail.com)

* Objetivo de la clase : Presentar como se trabaja computacionalemnte con modelos de lenguaje, de donde provienen y como se crean en si.

## Argumentos Base

* La ingeniería de AI consiste en 
    * Datos : Textos  no estrucutrados y multimodales
    * Procesos donde construir un contexto es lo importante , paso seguido hacer un prompt
    * Adaptación de los modelos (destilación) con fine tuning o eficiencia de los parámetros

* Los modelos core : codifican la información de manera que calcula matemáticamente que sencuiencia tiene mayor probabilidad de seguir en un texto:

    > Fue sin querer 

    > Fue sin querer **queriendo**

* AutoSupervision : Dado que los modelos requieren etiquetas para asi poder entrensarse, la estructura del lenguaje aprende prediciendo partes ocultas en los textyos crudos no etiquetados.

* La ingeniería de AI consiste en usar modelos fundacionales o vía API, donde :

    * Prompting -> Instrucciones
    * Rag -> El Contexto
    * Modelo -> Ajuste internos de los datos para entender como actuar de manera probabilistica

### Bases para que esto funcione de aquí en adelante
## Carácteristicas de un prompt 

Los prompts tanto para agentes como para Chatbots requieren de la siguiente ecuación , no la olvide, usela siempre :


* Role 
* Tarea
* Como debe ser el output
* Ejemplos (Nice to Have)
* Contexto

In [7]:
#######################
# ---- Librerias ---- #
#######################

import os
from openai import OpenAI
from dotenv import load_dotenv
import config  
from IPython.display import display, Markdown
from IPython.display import display, Markdown
client = OpenAI()

In [ ]:
from enum import Enum

class OpenAIModels(str, Enum):  # str -> valores como cadenas | Enum -> enumeración
    GPT_5_5 = "gpt-5.5"
    GPT_5_MINI = "gpt-5-mini"

MODEL = OpenAIModels.GPT_5_MINI

In [ ]:
def get_completion(system_prompt,           # Define el comportamiento del modelo
                   user_prompt,             # Solicitud del usuario
                   model=MODEL):
    # Los modelos gpt-5* NO aceptan `temperature` distinto del default (1)
    # ni `max_tokens` (usan `max_completion_tokens`). Por eso esta función
    # no pasa esos parámetros explícitamente.
    messages = [{"role": "user", "content": user_prompt}]
    if system_prompt is not None:
        messages = [{"role": "system", "content": system_prompt}, *messages]
    try:
        response = client.chat.completions.create(
            model=model,
            messages=messages,
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"An error occurred: {e}"

In [8]:
def display_responses(*args):
    """AYuda visual en formato markdown para comparar los modelos."""
    markdown_string = "<table><tr>"
    for arg in args:
        markdown_string += f"<th>System Prompt:<br />{arg['system_prompt']}<br /><br />"
        markdown_string += f"User Prompt:<br />{arg['user_prompt']}</th>"
    markdown_string += "</tr>"
    markdown_string += "<tr>"
    for arg in args:
        markdown_string += f"<td>Response:<br />{arg['response']}</td>"
    markdown_string += "</tr></table>"
    display(Markdown(markdown_string))

In [9]:
system_prompt = "Eres un experto en los 4 fantasticos, en comics y en historia de la segunda guerra mundial, tus respuestas son laconicas y coherentes y tienen un hilo conductor"
user_prompt = "Escribe un resumen sobre por qué Doom tiene conflicto con los 4 fantasticos"

print('=='*32)
print(f'Enviar solicitud al modelo:  {MODEL}')
baseline_response = get_completion(system_prompt, user_prompt)
print('=='*32)
display_responses({
    "system_prompt": system_prompt,
    "user_prompt": user_prompt, 
    "response": baseline_response
})

Enviar solicitud al modelo:  OpenAIModels.GPT_5_4_mini


<table><tr><th>System Prompt:<br />Eres un experto en los 4 fantasticos, en comics y en historia de la segunda guerra mundial, tus respuestas son laconicas y coherentes y tienen un hilo conductor<br /><br />User Prompt:<br />Escribe un resumen sobre por qué Doom tiene conflicto con los 4 fantasticos</th></tr><tr><td>Response:<br />Doom tiene conflicto con los 4 Fantásticos por una mezcla de orgullo, rivalidad y resentimiento.

Victor Von Doom ve a Reed Richards como su gran igual, e incluso como su principal rival intelectual. Ambos son genios, pero Doom cree que Reed le robó la gloria y que su accidente, el de Reed, fue una humillación injusta. A eso se suma que Doom no soporta no ser reconocido como el hombre más brillante y poderoso.

Además, los 4 Fantásticos representan todo lo que Doom desprecia: trabajo en equipo, altruismo y familia. Él prefiere el control absoluto y la superioridad personal. Por eso sus choques no son solo por planes de conquista, sino por una enemistad profundamente personal.

En resumen: Doom lucha contra los 4 Fantásticos porque los considera un obstáculo para su ambición, y porque Reed Richards es, para él, el símbolo de su mayor rivalidad y derrota.</td></tr></table>

### Transfomers 

In [12]:
import pandas as pd
import numpy as np
import tiktoken

In [ ]:
text = "Nunca es demasiado tarde para ser sabio."

# Los modelos gpt-5* usan el tokenizer `o200k_base`. tiktoken aún no
# mapea automáticamente todos los alias (ej. "gpt-5.5"), así que lo
# pedimos por nombre de encoding para que funcione con cualquier variante.
encoding = tiktoken.get_encoding("o200k_base")

token_ids = encoding.encode(text)
print(f"IDs de Tokens: {token_ids}")

tokens_text = [encoding.decode_single_token_bytes(t).decode("utf-8") for t in token_ids]
print(f"Tokens de texto: {tokens_text}")

df_tokens = pd.DataFrame({"Token ID": token_ids, "Token Text": tokens_text})
print(df_tokens)

In [16]:
response = client.embeddings.create(
    input=text,
    model="text-embedding-3-small"
)

embedding_vector = response.data[0].embedding
print(f"Dimensión del vector: {len(embedding_vector)}") 
print(f"Primeros 15 valores: {embedding_vector[:15]}")

Dimensión del vector: 1536
Primeros 15 valores: [0.049468994140625, 0.00640106201171875, -0.0230560302734375, 0.0285797119140625, 0.032318115234375, 0.005340576171875, -0.0027008056640625, 0.037109375, -0.03973388671875, 0.006008148193359375, 0.0162811279296875, -0.006717681884765625, 0.0200347900390625, -0.017974853515625, 0.0269317626953125]


In [ ]:
prompt = "La capital de Argentina es "

# Nota: gpt-5* NO permite `logprobs` (devuelve 403). Para esta demo de
# inspección de probabilidades usamos gpt-4o-mini, que sí lo soporta.
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": prompt}],
    max_tokens=15,
    logprobs=True,      # habilita la inspección de probabilidades
    top_logprobs=5,     # top-5 alternativas por token
)

generated_content = response.choices[0].message.content
print(f"Prompt: {prompt}")
print(f"Completado: {generated_content}\n")

for i, tok in enumerate(response.choices[0].logprobs.content):
    print(f"Token {i}: {tok.token!r}  logprob={tok.logprob:.4f}")
    for alt in tok.top_logprobs:
        print(f"    alt: {alt.token!r}  logprob={alt.logprob:.4f}")